# Hyperparameter Tuning

## Parameter vs. Hyperparameter

```
Parameter      = vom Modell GELERNT (Gewichte, Schwellenwerte)
Hyperparameter = von DIR gesetzt VOR dem Training
```

| Modell | Hyperparameter |
|--------|---------------|
| KNN | `n_neighbors` |
| Decision Tree | `max_depth`, `min_samples_split` |
| Random Forest | `n_estimators`, `max_depth`, `max_leaf_nodes` |

**Ziel:** Die besten Hyperparameter finden → bessere Modellperformance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings('ignore')

## Datensatz laden — Breast Cancer

In [ ]:
cancer = load_breast_cancer()

X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='label')

print(f"Datensatzgröße: {X.shape}")
print(f"Klassen: {cancer.target_names}")
print(f"Bösartig (0): {sum(y==0)} | Gutartig (1): {sum(y==1)}")
X.head()

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## Baseline: Random Forest ohne Tuning

In [ ]:
# Baseline — keine Hyperparameter gesetzt
rf_baseline = RandomForestClassifier(random_state=42)
rf_baseline.fit(X_train, y_train)

print(f"Baseline Accuracy: {rf_baseline.score(X_test, y_test):.4f}")

## Grid Search — Alle Kombinationen testen

Grid Search testet **alle möglichen Kombinationen** aus dem definierten Grid.

```
n_estimators:  [10, 100, 500]     → 3 Werte
max_depth:     [5, 10]            → 2 Werte  
max_leaf_nodes:[15, 30, 40]       → 3 Werte

3 × 2 × 3 = 18 Kombinationen
Mit cv=3  → 18 × 3 = 54 Trainingsläufe!
```

In [ ]:
# Schritt 1: Grid definieren
grid = {
    'n_estimators': [10, 100, 500],
    'max_depth': [5, 10],
    'max_leaf_nodes': [15, 30, 40]
}

# Schritt 2: GridSearchCV erstellen
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=grid,
    cv=3,
    verbose=1,
    n_jobs=-1
)

# Schritt 3: Trainieren
grid_search.fit(X_train, y_train)

In [ ]:
# Beste Hyperparameter
print("Beste Parameter:", grid_search.best_params_)
print(f"Bester CV-Score: {grid_search.best_score_:.4f}")
print(f"Test Accuracy:   {grid_search.score(X_test, y_test):.4f}")

## Random Search — Zufällige Kombinationen testen

Bei einem großen Parameterraum wäre Grid Search zu langsam.
Random Search testet nur **n_iter zufällige Kombinationen**.

In [ ]:
# Großer Parameterraum
random_grid = {
    'n_estimators': [int(x) for x in np.linspace(200, 2000, 10)],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [int(x) for x in np.linspace(10, 110, 11)] + [None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Mögliche Kombinationen:
total = 10 * 2 * 12 * 3 * 3 * 2
print(f"Mögliche Kombinationen: {total}")
print(f"Random Search testet nur: 15")

In [ ]:
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=random_grid,
    n_iter=15,     # nur 15 zufällige Kombinationen
    cv=3,
    n_jobs=-1,     # alle CPU-Kerne nutzen
    random_state=42
)

random_search.fit(X_train, y_train)

print("Beste Parameter:", random_search.best_params_)
print(f"Bester CV-Score: {random_search.best_score_:.4f}")
print(f"Test Accuracy:   {random_search.score(X_test, y_test):.4f}")

## Vergleich: Baseline vs. Grid Search vs. Random Search

In [ ]:
results = {
    'Baseline (kein Tuning)': rf_baseline.score(X_test, y_test),
    'Grid Search':            grid_search.score(X_test, y_test),
    'Random Search':          random_search.score(X_test, y_test)
}

print("Accuracy Vergleich:")
print("-" * 40)
for name, score in results.items():
    print(f"{name:<30} {score:.4f}")

# Balkendiagramm
plt.figure(figsize=(8, 4))
plt.bar(results.keys(), results.values(), color=['gray', 'steelblue', 'coral'])
plt.ylim(0.9, 1.0)
plt.title('Hyperparameter Tuning Vergleich')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()

## Bestes Modell wiederverwenden

Nach Grid/Random Search ist das beste Modell bereits trainiert — direkt nutzbar!

In [ ]:
# Bestes Modell aus Grid Search
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

## Wann welche Methode?

| Methode | Parameterraum | Geschwindigkeit | Garantiert optimal? |
|---------|--------------|-----------------|---------------------|
| **Grid Search** | Klein (< 100 Kombi.) | Langsam | Ja (aus Grid) |
| **Random Search** | Groß (> 100 Kombi.) | Schnell | Nein |

**Empfohlene Strategie:**
1. Random Search mit großem Grid → grob den besten Bereich finden
2. Grid Search in diesem Bereich → fein optimieren